In [ ]:
import pandas as pd

def _as_list(value):
    """把单个值统一转成列表，方便批量循环。"""
    if isinstance(value, (list, tuple, pd.Series, pd.Index)):
        return list(value)
    return [value]

def add_months(dt, months):
    """按 Excel EDATE 逻辑加月份：目标月没有对应日期时取月末。"""
    return pd.Timestamp(dt) + pd.DateOffset(months=months)
    
def pmt(rate, nper, pv):
    """返回等额本息每期还款额，对应 Excel: -PMT(rate, nper, pv)。"""
    if rate == 0:
        return pv / nper
    return pv * rate / (1 - (1 + rate) ** (-nper))
    
def xirr(cashflows, dates, low=-0.999999, high=10, tol=1e-10, max_iter=1000):
    """用二分法计算 XIRR，现金流需同时包含正负值。"""
    f_low = xnpv(low, cashflows, dates)
    f_high = xnpv(high, cashflows, dates)

    while f_low * f_high > 0 and high < 1_000:
        high *= 2
        f_high = xnpv(high, cashflows, dates)

    if f_low * f_high > 0:
        raise ValueError("XIRR 无法求解：请检查现金流是否同时包含正负值，或扩大 high。")

    for _ in range(max_iter):
        mid = (low + high) / 2
        f_mid = xnpv(mid, cashflows, dates)

        if abs(f_mid) < tol:
            return mid

        if f_low * f_mid < 0:
            high = mid
            f_high = f_mid
        else:
            low = mid
            f_low = f_mid

    return mid

def xnpv(rate, cashflows, dates):
    """按真实日期折现的净现值。"""
    d0 = pd.Timestamp(dates[0])
    return sum(
        cf / ((1 + rate) ** ((pd.Timestamp(d) - d0).days / 365))
        for cf, d in zip(cashflows, dates)
    )

def build_monthly_repay_dates(start_date, periods):
    """按起息日逐月生成还款日期，对齐上面 add_months/Excel EDATE 逻辑。"""
    start_date = pd.Timestamp(start_date)
    return [add_months(start_date, i) for i in range(1, periods + 1)]

def calc_repay_plan_by_target_xirr(
    principal,
    start_date,
    repay_dates,
    target_converted_annual_rate=0.24,
    weights=None,
):
    """
    根据本金、起息日、还款日期列表、目标年化，用折现系数闭式解计算还款计划金额。

    口径：先把目标年化转成对应 XIRR，再按真实累计计息天数折现：
        目标XIRR = (1 + 目标年化 / 12)^12 - 1
        折现系数 = 1 / ((1 + 目标XIRR)^(累计计息天数 / 365))

    参数：
    - principal: 本金，例如 10000
    - start_date: 0期起息日/放款日，例如 "2025-01-15"
    - repay_dates: 还款日期列表，例如 ["2025-02-15", "2025-03-15", ...]
    - target_converted_annual_rate: 目标 XIRR折算年化，默认 24%
    - weights: 每期还款权重，默认每期等额；如果传入 [1, 2, 3]，则每期金额按 1:2:3 分配

    返回：
    - summary: 汇总结果，含目标折现XIRR、折现系数合计、精确还款额等
    - schedule: 现金流明细，含累计计息天数、折现系数和每期还款金额
    """
    start_date = pd.Timestamp(start_date)
    repay_dates = [pd.Timestamp(d) for d in repay_dates]
    periods = len(repay_dates)

    if periods == 0:
        raise ValueError("repay_dates 不能为空。")
    if any(d <= start_date for d in repay_dates):
        raise ValueError("所有还款日期都必须晚于起息日。")

    if weights is None:
        weights = [1] * periods
    if len(weights) != periods:
        raise ValueError("weights 长度必须和 repay_dates 一致。")

    weights = pd.Series(weights, dtype="float64")
    if (weights <= 0).any():
        raise ValueError("weights 必须全部大于 0。")

    # 目标定义：converted_annual_rate = xirr_period_rate * 12
    # 且 xirr_period_rate = (1 + XIRR)^(1/12) - 1
    # 所以先由目标折算年化反推出需要用于真实日期折现的目标 XIRR。
    target_xirr_period_rate = target_converted_annual_rate / 12
    target_discount_xirr = (1 + target_xirr_period_rate) ** 12 - 1

    cumulative_days = [(d - start_date).days for d in repay_dates]
    discount_factors = [
        1 / ((1 + target_discount_xirr) ** (days / 365))
        for days in cumulative_days
    ]
    discount_factor_sum = sum(w * df for w, df in zip(weights, discount_factors))

    # 闭式解：令 -principal + base_amount * sum(weight_i * discount_factor_i) = 0。
    base_amount = principal / discount_factor_sum
    repay_amounts = [base_amount * w for w in weights]

    cashflow_dates = [start_date] + repay_dates
    cashflows = [-principal] + repay_amounts

    # 精确解已按目标 XIRR 折现求出，不再用二分法反算 XIRR。
    annual_xirr = target_discount_xirr
    xirr_period_rate = target_xirr_period_rate
    converted_annual_rate = target_converted_annual_rate
    diff = 0.0

    summary = pd.DataFrame(
        {
            "0期起息日": [start_date.date()],
            "本金": [principal],
            "期数": [periods],
            "年化费率": [target_converted_annual_rate],
            "月度费率": [target_xirr_period_rate],
            "折现系数合计": [discount_factor_sum],
            "基准还款额": [base_amount],
            "反算XIRR": [annual_xirr],
            "XIRR折算周期费率": [xirr_period_rate],
            "XIRR折算年化": [converted_annual_rate],
            "折算年化与目标差": [diff],
            "折算年化与目标差_bp": [diff * 10_000],
        }
    )

    schedule = pd.DataFrame(
        {
            "期次": range(0, periods + 1),
            "日期": [d.date() for d in cashflow_dates],
            "累计计息天数": [0] + cumulative_days,
            "折现系数": [1] + discount_factors,
            "现金流": cashflows,
            "还款金额": [0] + repay_amounts,
        }
    )

    return summary, schedule


def calc_rate_compare_row(
    principal,
    start_date,
    periods,
    annual_rate,
    actual_rate,
    rate_label=None,
):
    """
    单组参数输出一行汇总：
    - 年利率PMT还款值
    - 实际利率PMT还款值
    - XIRR折算年化逼近年利率的还款值
    - XIRR折算年化逼近实际利率的还款值
    - 两个XIRR计划累计额分别与年利率PMT累计额的差
    """
    start_date = pd.Timestamp(start_date)
    repay_dates = build_monthly_repay_dates(start_date, periods)

    annual_pmt_amount = pmt(annual_rate / 12, periods, principal)
    actual_pmt_amount = pmt(actual_rate / 12, periods, principal)

    annual_xirr_summary, _ = calc_repay_plan_by_target_xirr(
        principal=principal,
        start_date=start_date,
        repay_dates=repay_dates,
        target_converted_annual_rate=annual_rate,
    )
    actual_xirr_summary, _ = calc_repay_plan_by_target_xirr(
        principal=principal,
        start_date=start_date,
        repay_dates=repay_dates,
        target_converted_annual_rate=actual_rate,
    )

    annual_xirr_amount = annual_xirr_summary["基准还款额"].iloc[0]
    actual_xirr_amount = actual_xirr_summary["基准还款额"].iloc[0]
    annual_discount_xirr = annual_xirr_summary["反算XIRR"].iloc[0]
    actual_discount_xirr = actual_xirr_summary["反算XIRR"].iloc[0]
    annual_discount_factor_sum = annual_xirr_summary["折现系数合计"].iloc[0]
    actual_discount_factor_sum = actual_xirr_summary["折现系数合计"].iloc[0]

    annual_pmt_total = annual_pmt_amount * periods
    actual_pmt_total = actual_pmt_amount * periods
    annual_xirr_total = annual_xirr_amount * periods
    actual_xirr_total = actual_xirr_amount * periods

    return {
        "本金": principal,
        "起息日": start_date.date(),
        "期限": periods,
        "首期还款日": repay_dates[0].date(),
        "末期还款日": repay_dates[-1].date(),
        "计息总天数": (repay_dates[-1] - start_date).days,
        "利率组合": rate_label if rate_label is not None else f"{annual_rate:.2%}/{actual_rate:.2%}",
        "年利率": annual_rate,
        "资方利率": actual_rate,
        "年利率PMT还款值": annual_pmt_amount,
        "资方利率PMT还款值": actual_pmt_amount,
        "年利率对应XIRR": annual_discount_xirr,
        "资方利率对应XIRR": actual_discount_xirr,
        "年利率折现系数合计": annual_discount_factor_sum,
        "资方利率折现系数合计": actual_discount_factor_sum,
        "XIRR转化年化逼近年利率还款值": annual_xirr_amount,
        "XIRR转化年化逼近资方利率还款值": actual_xirr_amount,
        "年利率PMT累计值": annual_pmt_total,
        "资方利率PMT累计值": actual_pmt_total,
        "XIRR逼近年利率累计值": annual_xirr_total,
        "XIRR逼近资方利率累计值": actual_xirr_total,
        "XIRR逼近年利率累计-年利率PMT累计": annual_xirr_total - annual_pmt_total,
        "XIRR逼近资方利率累计-资方利率PMT累计": actual_xirr_total - actual_pmt_total,
    }


def batch_calc_rate_compare(
    principal=10_000,
    start_dates="2025-02-15",
    periods=12,
    rate_pairs=None,
):
    """
    批量汇总输出，适合直接拿去做透视表。

    参数示例：
    rate_pairs = [
        {"年利率": 0.24, "资方利率": 0.06},
        {"年利率": 0.36, "资方利率": 0.09},
    ]
    start_dates = ["2025-02-15", "2025-03-15"]
    periods = [3, 6, 12]
    """
    if rate_pairs is None:
        rate_pairs = [{"年利率": 0.24, "资方利率": 0.06}]
    elif isinstance(rate_pairs, dict):
        rate_pairs = [rate_pairs]
    elif isinstance(rate_pairs, tuple) and len(rate_pairs) >= 2 and not isinstance(rate_pairs[0], (dict, list, tuple)):
        rate_pairs = [rate_pairs]

    rows = []
    for start_date in _as_list(start_dates):
        for period in _as_list(periods):
            for pair in rate_pairs:
                if isinstance(pair, dict):
                    annual_rate = pair["年利率"]
                    actual_rate = pair["资方利率"]
                    rate_label = pair.get("利率组合")
                else:
                    annual_rate, actual_rate = pair[:2]
                    rate_label = pair[2] if len(pair) > 2 else None

                rows.append(
                    calc_rate_compare_row(
                        principal=principal,
                        start_date=start_date,
                        periods=int(period),
                        annual_rate=float(annual_rate),
                        actual_rate=float(actual_rate),
                        rate_label=rate_label,
                    )
                )

    return pd.DataFrame(rows)

In [ ]:
# # 示例1：单个起息日
# summary, schedule = calc_repay_plan_by_target_xirr(
#     principal=10_000,
#     start_date="2025-02-15",
#     repay_dates = build_monthly_repay_dates("2025-02-15", 12),
#     target_converted_annual_rate=0.24,
# )

# display(summary)
# display(schedule)


,0期起息日,本金,期数,目标XIRR折算年化,目标XIRR折算周期费率,目标折现XIRR,折现系数合计,基准还款额,反算XIRR,XIRR折算周期费率,XIRR折算年化,折算年化与目标差,折算年化与目标差_bp
0,2025-02-15,10000,12,0.24,0.02,0.268242,10.585113,944.723069,0.268242,0.02,0.24,0.0,0.0


,期次,日期,累计计息天数,折现系数,现金流,还款金额
0,0,2025-02-15,0,1.000000,-10000.000000,0.000000
1,1,2025-03-15,28,0.981936,944.723069,944.723069
2,2,2025-04-15,59,0.962317,944.723069,944.723069
3,3,2025-05-15,89,0.943704,944.723069,944.723069
4,4,2025-06-15,120,0.924848,944.723069,944.723069
5,5,2025-07-15,150,0.906960,944.723069,944.723069
6,6,2025-08-15,181,0.888839,944.723069,944.723069
7,7,2025-09-15,212,0.871080,944.723069,944.723069
8,8,2025-10-15,242,0.854232,944.723069,944.723069
9,9,2025-11-15,273,0.837164,944.723069,944.723069


In [ ]:
# 示例：你只需要改这里的输入，就能得到透视表用的汇总 dataframe
rate_pairs = [
    {"年利率": 0.2400, "资方利率": 0.0650},
    {"年利率": 0.2400, "资方利率": 0.0613},
    {"年利率": 0.2400, "资方利率": 0.0600},
    {"年利率": 0.2400, "资方利率": 0.0583},
    {"年利率": 0.2400, "资方利率": 0.0500},
    {"年利率": 0.2400, "资方利率": 0.0480},
    {"年利率": 0.2400, "资方利率": 0.0450},
    {"年利率": 0.2400, "资方利率": 0.0400},
    {"年利率": 0.2400, "资方利率": 0.0360},
]

result_df = batch_calc_rate_compare(
    principal=10_000,
    start_dates=["2025-02-15","2025-07-15",'2025-09-15'],
    periods=[12,6,3,1],
    rate_pairs=rate_pairs,
)

# result_df.to_excel("result.xlsx", index=False)
with pd.ExcelWriter("result.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    result_df.to_excel(writer, sheet_name="by月", index=False)

In [6]:
# 示例：你只需要改这里的输入，就能得到透视表用的汇总 dataframe
rate_pairs = [
    {"年利率": 0.2400, "资方利率": 0.0600},
]

result_df = batch_calc_rate_compare(
    principal=10_000,
    # start_dates=["2025-01-15","2025-02-15",
    #             "2025-03-15","2025-04-15",
    #             "2025-05-15","2025-06-15",
    #             "2025-07-15","2025-08-15",
    #             "2025-09-15","2025-10-15",
    #             "2025-11-15","2025-12-15",],
    start_dates = pd.date_range("2025-01-01", "2025-12-31", freq="D"),
    periods=[12,6,3,1],
    rate_pairs=rate_pairs,
)

# result_df.to_excel("result.xlsx", index=False)
with pd.ExcelWriter("D:\\10.LTV月度更新\\息费测算\\新规计息对比老方法.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    result_df.to_excel(writer, sheet_name="by月", index=False)